<a href="https://colab.research.google.com/github/RegiRezende/MathCode/blob/Python_codes/DECOMP_QR_G_Schmidt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import math

class Matrix:
    def __init__(self, n_col, n_row, values=None, two_d_array=None):
        self.n_col = n_col
        self.n_row = n_row

        if two_d_array is not None:
            self.values = two_d_array
        elif values is not None:
            self.values = values
        else:
            self.values = [[0.0] * n_col for _ in range(n_row)]

    def transpose(self):
        """Retorna a matriz transposta"""
        transposed = [[self.values[j][i] for j in range(self.n_row)]
                      for i in range(self.n_col)]
        return Matrix(self.n_row, self.n_col, two_d_array=transposed)

    def multiply(self, other):
        """Multiplica esta matriz por outra"""
        if self.n_col != other.n_row:
            raise ValueError("Dimensões incompatíveis para multiplicação")

        result = [[0.0] * other.n_col for _ in range(self.n_row)]
        for i in range(self.n_row):
            for j in range(other.n_col):
                for k in range(self.n_col):
                    result[i][j] += self.values[i][k] * other.values[k][j]

        return Matrix(self.n_row, other.n_col, two_d_array=result)

    @staticmethod
    def proj_on(u, v):
        """Calcula a projeção de u em v"""
        dot_uv = sum(u[i] * v[i] for i in range(len(u)))
        dot_vv = sum(v[i] * v[i] for i in range(len(v)))

        if dot_vv == 0:
            return [0.0] * len(u)

        scalar = dot_uv / dot_vv
        return [scalar * vi for vi in v]

    @staticmethod
    def length(v):
        """Calcula o comprimento (norma) de um vetor"""
        return math.sqrt(sum(x * x for x in v))


def QR_GS(M: Matrix) -> tuple:
    """
    Decomposição QR usando o processo de Gram-Schmidt

    Args:
        M: Matriz a ser decomposta

    Returns:
        Tupla (Q, R) onde Q é ortogonal e R é triangular superior
    """
    Q_vectors = []

    for i in range(M.n_col):
        # Obtém o vetor coluna i
        curr_column = [M.values[row][i] for row in range(M.n_row)]

        # Inicializa q com o vetor atual
        q = curr_column.copy()

        # Subtrai as projeções nos vetores ortogonais já calculados
        for k in range(i):
            proj = Matrix.proj_on(curr_column, Q_vectors[k])
            for j in range(len(q)):
                q[j] -= proj[j]

        # Normaliza o vetor
        length = Matrix.length(q)
        if length > 1e-10:  # Evita divisão por zero
            q = [x / length for x in q]
        else:
            # Vetor zero - não deve acontecer para matrizes de posto completo
            q = [0.0] * len(q)

        # Armazena o vetor
        Q_vectors.append(q)

    # Constrói a matriz Q (cada vetor em Q_vectors é uma coluna)
    Q_data = [[Q_vectors[j][i] for j in range(len(Q_vectors))]
              for i in range(len(Q_vectors[0]))]
    Q = Matrix(n_col=M.n_col, n_row=M.n_row, two_d_array=Q_data)

    # Calcula R = Q^T * A
    Q_T = Q.transpose()
    R = Q_T.multiply(M)

    return (Q, R)


# Exemplo de uso
if __name__ == "__main__":
    # Cria uma matriz exemplo
    A_data = [
        [1.0, 1.0, 0.0],
        [1.0, 0.0, 1.0],
        [0.0, 1.0, 1.0]
    ]

    A = Matrix(n_col=3, n_row=3, two_d_array=A_data)

    # Calcula a decomposição QR
    Q, R = QR_GS(A)

    # Verifica se Q^T * Q = I
    Q_T = Q.transpose()
    QT_Q = Q_T.multiply(Q)

    print("Matriz A:")
    for row in A.values:
        print(row)

    print("\nMatriz Q:")
    for row in Q.values:
        print(row)

    print("\nMatriz R:")
    for row in R.values:
        print(row)

    print("\nQ^T * Q (deve ser aproximadamente identidade):")
    for row in QT_Q.values:
        print([round(x, 5) for x in row])

    print("\nQ * R (deve ser aproximadamente A):")
    QR = Q.multiply(R)
    for row in QR.values:
        print([round(x, 5) for x in row])

Matriz A:
[1.0, 1.0, 0.0]
[1.0, 0.0, 1.0]
[0.0, 1.0, 1.0]

Matriz Q:
[0.7071067811865475, 0.4082482904638631, -0.5773502691896256]
[0.7071067811865475, -0.4082482904638631, 0.5773502691896256]
[0.0, 0.8164965809277261, 0.5773502691896257]

Matriz R:
[1.414213562373095, 0.7071067811865475, 0.7071067811865475]
[0.0, 1.2247448713915892, 0.4082482904638631]
[0.0, 1.1102230246251565e-16, 1.1547005383792515]

Q^T * Q (deve ser aproximadamente identidade):
[1.0, 0.0, 0.0]
[0.0, 1.0, 0.0]
[0.0, 0.0, 1.0]

Q * R (deve ser aproximadamente A):
[1.0, 1.0, 0.0]
[1.0, -0.0, 1.0]
[0.0, 1.0, 1.0]
